# Test Statistici Formali - Significatività delle Differenze

Questo notebook completa l'analisi aggiungendo test statistici formali mancanti:
- t-test per differenze di genere
- ANOVA per differenze geografiche
- Chi-square per indipendenza cluster-genere
- Assunzioni del modello lineare

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, f_oneway, chi2_contingency, shapiro, levene, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Configurazione
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
%matplotlib inline

# Impostare seed per riproducibilità
np.random.seed(42)

## 1. Caricamento Dati

In [ ]:
# Carica i dati cleanati dal notebook pulizia
import mysql.connector
from mysql.connector import Error

try:
    connection = mysql.connector.connect(
        host='localhost',
        database='project_work',
        user='root',
        password='Gaia123'
    )
    
    if connection.is_connected():
        cursor = connection.cursor()
        cursor.execute("SELECT * FROM clean_data")
        
        columns = [description[0] for description in cursor.description]
        data = cursor.fetchall()
        
        df = pd.DataFrame(data, columns=columns)
        print(f"Dataset caricato: {df.shape[0]} righe, {df.shape[1]} colonne")
        print(f"\nPrime colonne: {df.columns.tolist()[:10]}")
        
        cursor.close()
        
except Error as e:
    print(f"Errore connessione: {e}")
    print("\nCaricamento alternativo da CSV...")
    # Fallback: caricare da CSV se disponibile
    df = pd.read_csv('../data/clean_data.csv')
    
finally:
    if connection.is_connected():
        connection.close()

## 2. Test t per Differenze di Genere

Verifichiamo se le differenze tra maschi e femmine in ansia, coping, resilienza sono statisticamente significative.

In [ ]:
# Preparazione: assicurati che le variabili siano presenti
# Se gli indici psicologici non sono pre-calcolati, calcolali

# Verifica le colonne per genere
gender_col = [col for col in df.columns if 'genere' in col.lower() or 'gender' in col.lower()][0] if any('genere' in col.lower() for col in df.columns) else None

if gender_col is None:
    print("Colonna genere non trovata")
    print("Colonne disponibili:", df.columns.tolist())
else:
    print(f"Colonna genere trovata: {gender_col}")
    print(f"\nValori unici: {df[gender_col].unique()}")

In [ ]:
# T-test indipendenti per ogni variabile psicologica
# Assumendo che esitano colonne per: ansia_totale, coping, resilienza, evitamento, gap_psicologico

psych_variables = ['ansia_totale', 'coping', 'resilienza', 'evitamento', 'gap_psicologico']
gender_col = 'genere'  # Adatta in base al tuo dataset

# Filtra le variabili che effettivamente esistono
psych_variables = [v for v in psych_variables if v in df.columns]

print(f"Variabili psicologiche trovate: {psych_variables}")
print(f"\n{'='*70}")
print("T-TEST INDIPENDENTI: DIFFERENZE TRA GENERI")
print(f"{'='*70}\n")

results_ttest = []

for var in psych_variables:
    # Separa per genere (assumi 'M' e 'F' o 0/1)
    group1 = df[df[gender_col].isin([0, 'M', 'Maschio', 'M'])][var].dropna()
    group2 = df[df[gender_col].isin([1, 'F', 'Femmina', 'F'])][var].dropna()
    
    if len(group1) > 0 and len(group2) > 0:
        # T-test
        t_stat, p_value = ttest_ind(group1, group2)
        
        mean_g1 = group1.mean()
        mean_g2 = group2.mean()
        std_g1 = group1.std()
        std_g2 = group2.std()
        
        # Effetto Cohen's d
        cohens_d = (mean_g1 - mean_g2) / np.sqrt(((len(group1)-1)*std_g1**2 + (len(group2)-1)*std_g2**2) / (len(group1) + len(group2) - 2))
        
        # Significatività
        sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
        
        print(f"Variabile: {var}")
        print(f"  Genere M: μ = {mean_g1:.3f} ± {std_g1:.3f} (n={len(group1)})")
        print(f"  Genere F: μ = {mean_g2:.3f} ± {std_g2:.3f} (n={len(group2)})")
        print(f"  t({len(group1)+len(group2)-2}) = {t_stat:.3f}, p = {p_value:.4f} {sig}")
        print(f"  Cohen's d = {cohens_d:.3f}")
        print()
        
        results_ttest.append({
            'Variable': var,
            'Mean_M': mean_g1,
            'SD_M': std_g1,
            'N_M': len(group1),
            'Mean_F': mean_g2,
            'SD_F': std_g2,
            'N_F': len(group2),
            't_stat': t_stat,
            'p_value': p_value,
            'Cohens_d': cohens_d,
            'Significant': sig != 'ns'
        })

df_ttest = pd.DataFrame(results_ttest)
print(f"\n{df_ttest.to_string(index=False)}")

## 3. ANOVA per Differenze Geografiche (Macro-aree)

Test se l'ansia varia significativamente tra Nord, Centro, Sud.

In [ ]:
# Identifica colonna area geografica
geo_col = [col for col in df.columns if 'area' in col.lower() or 'macro' in col.lower() or 'regione' in col.lower()]

if geo_col:
    geo_col = geo_col[0]
    print(f"Colonna geografica: {geo_col}")
    print(f"Aree uniche: {df[geo_col].unique()}")
else:
    print("Colonna geografica non trovata. Colonne disponibili:")
    print(df.columns.tolist())

In [ ]:
# ANOVA one-way per aree geografiche
print(f"{'='*70}")
print("ANOVA: DIFFERENZE GEOGRAFICHE (MACRO-AREE)")
print(f"{'='*70}\n")

results_anova = []

for var in psych_variables:
    # Separa per area geografica
    groups = []
    area_names = []
    
    for area in df[geo_col].unique():
        if pd.notna(area):
            group_data = df[df[geo_col] == area][var].dropna()
            if len(group_data) > 0:
                groups.append(group_data)
                area_names.append(str(area))
    
    if len(groups) >= 2:
        # ANOVA
        f_stat, p_value = f_oneway(*groups)
        
        # Eta-squared (effect size)
        grand_mean = np.concatenate(groups).mean()
        ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
        ss_total = sum((x - grand_mean)**2 for g in groups for x in g)
        eta_squared = ss_between / ss_total if ss_total > 0 else 0
        
        sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
        
        print(f"Variabile: {var}")
        for i, (area, group) in enumerate(zip(area_names, groups)):
            print(f"  {area}: μ = {group.mean():.3f} ± {group.std():.3f} (n={len(group)})")
        print(f"  F({len(groups)-1},{sum(len(g) for g in groups)-len(groups)}) = {f_stat:.3f}, p = {p_value:.4f} {sig}")
        print(f"  η² = {eta_squared:.4f}")
        print()
        
        results_anova.append({
            'Variable': var,
            'F_stat': f_stat,
            'p_value': p_value,
            'Eta_squared': eta_squared,
            'Significant': sig != 'ns'
        })

df_anova = pd.DataFrame(results_anova)
print(f"\n{df_anova.to_string(index=False)}")

## 4. Chi-Square Test: Indipendenza Cluster × Genere

In [ ]:
# Verifica se esiste colonna cluster
cluster_col = [col for col in df.columns if 'cluster' in col.lower()]

if cluster_col:
    cluster_col = cluster_col[0]
    print(f"Colonna cluster: {cluster_col}")
    print(f"Cluster unici: {sorted(df[cluster_col].unique())}")
    
    # Chi-square test
    contingency = pd.crosstab(df[cluster_col], df[gender_col])
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    
    print(f"\n{'='*70}")
    print("CHI-SQUARE TEST: INDIPENDENZA CLUSTER × GENERE")
    print(f"{'='*70}\n")
    
    print("Tabella di contingenza:")
    print(contingency)
    print(f"\nχ²({dof}) = {chi2:.3f}, p = {p_value:.4f}", "***" if p_value < 0.05 else "ns")
    
    # Cramér's V
    n = contingency.sum().sum()
    min_dim = min(contingency.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
    print(f"Cramér's V = {cramers_v:.4f}")
else:
    print("Colonna cluster non trovata")

## 5. Verifica Assunzioni Modello Lineare

- Normalità residui (Shapiro-Wilk)
- Omoschedasticità (Levene test)
- Correlazioni significative

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

print(f"{'='*70}")
print("VERIFICA ASSUNZIONI MODELLO LINEARE")
print(f"{'='*70}\n")

# Costruisci un modello semplice: ansia_totale ~ coping + resilienza + evitamento
X = df[['coping', 'resilienza', 'evitamento']].dropna()
y = df.loc[X.index, 'ansia_totale']

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)
residuals = y - y_pred

# 1. Normalità dei residui (Shapiro-Wilk)
print("1. TEST DI NORMALITÀ (Shapiro-Wilk)")
print("-" * 50)
stat, p = shapiro(residuals)
print(f"W = {stat:.4f}, p = {p:.4f}")
if p > 0.05:
    print("✓ Residui approssimativamente normali (p > 0.05)")
else:
    print("✗ Residui NON normali (p < 0.05) - violazione assunzione")

# 2. Omoschedasticità (Levene test sugli errori)
print(f"\n2. TEST OMOSCHEDASTICITÀ (Levene)")
print("-" * 50)
# Dividi residui per quartili di valori predetti
quartiles = pd.qcut(y_pred, q=4, duplicates='drop')
groups_residuals = [residuals[quartiles == q] for q in quartiles.unique()]
stat, p = levene(*groups_residuals)
print(f"Levene stat = {stat:.4f}, p = {p:.4f}")
if p > 0.05:
    print("✓ Varianze omogenee (p > 0.05)")
else:
    print("✗ Varianze NON omogenee (p < 0.05) - eteroschedasticità")

# 3. R² del modello
print(f"\n3. PERFORMANCE MODELLO")
print("-" * 50)
r2 = model.score(X, y)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f"R² = {r2:.4f}")
print(f"RMSE = {rmse:.4f}")
print(f"Coefficienti: Coping = {model.coef_[0]:.4f}, Resilienza = {model.coef_[1]:.4f}, Evitamento = {model.coef_[2]:.4f}")

# Visualizza residui
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[0])
axes[0].set_title("Q-Q Plot (Normalità Residui)")

# Residuals vs Fitted
axes[1].scatter(y_pred, residuals, alpha=0.5)
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_xlabel("Valori Predetti")
axes[1].set_ylabel("Residui")
axes[1].set_title("Residui vs Fitted (Omoschedasticità)")

plt.tight_layout()
plt.show()

## 6. Significatività Correlazioni (Pearson con p-values)

In [ ]:
print(f"{'='*70}")
print("CORRELAZIONI CON TEST DI SIGNIFICATIVITÀ")
print(f"{'='*70}\n")

from scipy.stats import pearsonr

# Matrice correlazioni con p-values
corr_data = df[psych_variables].dropna()

corr_matrix = corr_data.corr()
pvalue_matrix = pd.DataFrame(np.zeros_like(corr_matrix), columns=corr_matrix.columns, index=corr_matrix.index)

for i, col1 in enumerate(corr_matrix.columns):
    for j, col2 in enumerate(corr_matrix.columns):
        if i <= j:
            if i == j:
                pvalue_matrix.iloc[i, j] = 0
            else:
                _, p = pearsonr(corr_data[col1], corr_data[col2])
                pvalue_matrix.iloc[i, j] = p
                pvalue_matrix.iloc[j, i] = p

print("Matrice Correlazioni (Pearson):")
print(corr_matrix.round(3))

print("\nMatrice p-values:")
print(pvalue_matrix.round(4))

print("\nCorrelazioni significative (p < 0.05):")
for i, col1 in enumerate(corr_matrix.columns):
    for j, col2 in enumerate(corr_matrix.columns):
        if i < j and pvalue_matrix.iloc[i, j] < 0.05:
            sig = "***" if pvalue_matrix.iloc[i, j] < 0.001 else "**" if pvalue_matrix.iloc[i, j] < 0.01 else "*"
            print(f"  {col1} ↔ {col2}: r = {corr_matrix.iloc[i, j]:.3f}, p = {pvalue_matrix.iloc[i, j]:.4f} {sig}")

## 7. Sommario Risultati Statistici

In [ ]:
print(f"\n\n{'='*70}")
print("SOMMARIO RISULTATI TEST STATISTICI")
print(f"{'='*70}")

print("\n1. DIFFERENZE TRA GENERI (t-test):")
sig_gender = df_ttest[df_ttest['Significant']]
if len(sig_gender) > 0:
    print(f"   Variabili significative: {list(sig_gender['Variable'])}")
else:
    print("   Nessuna variabile significativamente diversa per genere")

print("\n2. DIFFERENZE GEOGRAFICHE (ANOVA):")
sig_geo = df_anova[df_anova['Significant']]
if len(sig_geo) > 0:
    print(f"   Variabili significative: {list(sig_geo['Variable'])}")
else:
    print("   Nessuna variabile significativamente diversa per area geografica")

print("\n3. INDIPENDENZA CLUSTER × GENERE:")
try:
    if p_value < 0.05:
        print(f"   Cluster e genere NON sono indipendenti (χ² = {chi2:.3f}, p = {p_value:.4f})")
    else:
        print(f"   Cluster e genere sono indipendenti (χ² = {chi2:.3f}, p = {p_value:.4f})")
except:
    print("   Test non eseguito (colonna cluster non trovata)")

print("\n4. ASSUNZIONI MODELLO LINEARE:")
print(f"   Normalità residui: {'✓ OK' if p > 0.05 else '✗ Violata'}")
print(f"   Omoschedasticità: {'✓ OK' if p > 0.05 else '✗ Violata'}")
print(f"   R² modello: {r2:.4f}")

print(f"\n{'='*70}")